# Assignment 2: Implement Scaled Dot-Product Attention and a Mini Transformer Encoder
**Objective:** Implement a Transformer Encoder from scratch using only PyTorch and test it on a text classification task.

### Components Implemented:
1. Scaled Dot-Product Attention
2. Multi-Head Self-Attention
3. Positional Encoding
4. Feed Forward Network
5. Transformer Encoder Block
6. Text Classifier using Transformer Encoder

In [1]:
# Cell 1: Import Libraries and Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import math
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import Counter

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [2]:
# Cell 2: Dataset, Augmentation, and Preprocessing
# ── Custom toy sentiment dataset ──────────────────────────────────────────────
raw_data = [
    # Positive (label = 1)
    ("I absolutely loved this movie it was fantastic", 1),
    ("The food was delicious and the service was great", 1),
    ("What an amazing experience I would highly recommend", 1),
    ("This product exceeded all my expectations wonderful", 1),
    ("The concert was brilliant and the musicians were outstanding", 1),
    ("I am so happy with my purchase it works perfectly", 1),
    ("Excellent quality and fast delivery very satisfied", 1),
    ("The book was captivating I could not put it down", 1),
    ("Beautiful scenery and wonderful weather on our trip", 1),
    ("The staff was friendly and the hotel was superb", 1),
    ("Outstanding performance by the entire cast loved it", 1),
    ("The app is intuitive and works flawlessly great job", 1),
    ("Incredibly tasty meal at this restaurant will return", 1),
    ("Perfect gift very pleased with the quality and packaging", 1),
    ("The workshop was informative and the trainer was brilliant", 1),
    # Negative (label = 0)
    ("This movie was terrible and a complete waste of time", 0),
    ("The food was awful and the service was very slow", 0),
    ("I hated this experience would never come back again", 0),
    ("The product broke after one day very disappointing", 0),
    ("Horrible concert the sound quality was dreadful", 0),
    ("I regret buying this completely useless and frustrating", 0),
    ("Poor quality and took forever to arrive very unhappy", 0),
    ("The book was boring and I could not finish it", 0),
    ("Terrible weather and crowded beaches ruined the trip", 0),
    ("The staff was rude and the room was dirty awful", 0),
    ("Dreadful performance by the cast deeply disappointing", 0),
    ("The app crashes constantly and is full of bugs horrible", 0),
    ("Disgusting food I got sick never eating there again", 0),
    ("Wrong item delivered packaging was damaged very upset", 0),
    ("The workshop was confusing and the trainer was unprepared", 0),
]

# --- Augmenting the dataset with more examples ------------------------------
# Expanded lists of positive and negative sentiment words/phrases for augmentation
positive_words_aug = [
    "loved", "fantastic", "amazing", "wonderful", "excellent", "brilliant",
    "happy", "perfect", "superb", "outstanding", "intuitive", "tasty",
    "pleased", "informative", "delightful", "extraordinary", "masterpiece",
    "breathtaking", "impressed", "efficient", "gem", "recommend", "great",
    "stunning", "delicious", "charming", "impressive", "joyful", "glorious",
    "magnificent", "splendid", "fabulous", "gorgeous", "heavenly", "ideal",
    "marvelous", "miraculous", "phenomenal", "pleasing", "resplendent",
    "sensational", "sparkling", "spectacular", "sublime", "super",
    "thrilling", "top-notch", "tremendous", "unforgettable", "uplifting",
    "vibrant", "victorious", "visionary", "welcoming", "whimsical",
    "wondrous", "zealous", "joy", "bliss", "cheer", "ecstatic", "optimistic",
    "positive", "radiant", "satisfied", "serene", "sincere", "smart",
    "successful", "sunny", "superb", "terrific", "thankful", "thoughtful",
    "thriving", "true", "trustworthy", "ultimate", "unbelievable", "unique",
    "united", "universal", "unmatched", "unreal", "unwavering", "upbeat",
    "valuable", "versatile", "victorious", "vigorous", "virtuous", "vivid",
    "warm", "wealthy", "well-done", "wholehearted", "wise", "witty"
]

negative_words_aug = [
    "terrible", "awful", "hated", "broke", "horrible", "useless", "poor",
    "boring", "dreadful", "rude", "dirty", "disappointing", "frustrating",
    "crashes", "sick", "damaged", "unprepared", "slow", "disgusting",
    "waste", "ruined", "never", "worst", "buggy", "lousy", "inferior",
    "bad", "unacceptable", "unpleasant", "unreliable", "deficient",
    "defective", "dire", "dismal", "distressing", "faulty", "flawed",
    "ghastly", "grim", "gruesome", "hideous", "lamentable", "miserable",
    "nasty", "nauseating", "offensive", "repellent", "revolting",
    "shabby", "shameful", "shocking", "sickening", "squalid", "stagnant",
    "unattractive", "unsatisfactory", "vile", "woeful", "anger", "anxious",
    "appalled", "apprehensive", "arrogant", "ashamed", "atrocious",
    "baffled", "bewildered", "bitter", "blatant", "bleak", "callous",
    "chaotic", "coarse", "cold", "confused", "crude", "cruel", "cynical",
    "deceitful", "deceptive", "dejected", "depraved", "despair", "detestable",
    "difficult", "disgruntled", "dishonest", "disloyal", "dismal", "disturbed",
    "drab", "dread", "dull", "empty", "envious", "erratic", "evil", "fake"
]

positive_templates_aug = [
    "I {} this experience it was {}",
    "The {} was {} and the outcome was {}",
    "What an {} opportunity I {} recommend",
    "This solution {} my expectations {}",
    "The performance was {} and the cast was {}",
    "I feel so {} with my results it's {}",
    "{} product and {} delivery very {}",
    "The story was {} I could not stop reading",
    "{} views and {} weather for our trip",
    "The assistance was {} and the support was {}",
    "An {} achievement, truly {} it",
    "The interface is {} and it works {} {} work",
    "Truly {} cuisine at this place will {}",
    "{} purchase, very {} with the item and packaging",
    "The training was {} and the instructor was {}",
    "A really {} and {} discovery",
    "This is absolutely {} and a must-see",
    "Everything about this was {} and {}"
]

negative_templates_aug = [
    "This experience was {} and a total {} of resources",
    "The {} was {} and the results were very {}",
    "I {} this situation would {} try it {}",
    "The service {} within a day very {}",
    "{} event the organization was {}",
    "I {} relying on this completely {} and {}",
    "{} design and took ages to fix very {}",
    "The dialogue was {} and I couldn't connect",
    "{} conditions and {} roads {} the journey",
    "The response was {} and the information was {} {}",
    "{} failure by the team deeply {}",
    "The system {} constantly and is full of {} {}",
    "{} quality I felt {} {} returning there again",
    "{} service given the item was {} very {}",
    "The session was {} and the speaker was {}",
    "A truly {} and {} outcome",
    "Everything about this was {} and {}"
]

# Generate more diverse sentences to reach 250 total (200 train, 50 test)
# We have 30 initial samples. We need 220 more. So, 110 positive and 110 negative.
new_positive_sentences = []
for _ in range(110): # Generate 110 new positive sentences
    template = random.choice(positive_templates_aug)
    num_fillers = template.count('{}')
    fillers = random.sample(positive_words_aug, num_fillers)
    new_positive_sentences.append(template.format(*fillers))

new_negative_sentences = []
for _ in range(110): # Generate 110 new negative sentences
    template = random.choice(negative_templates_aug)
    num_fillers = template.count('{}')
    fillers = random.sample(negative_words_aug, num_fillers)
    new_negative_sentences.append(template.format(*fillers))

# Append to existing raw_data
for sent in new_positive_sentences:
    raw_data.append((sent, 1))
for sent in new_negative_sentences:
    raw_data.append((sent, 0))

random.shuffle(raw_data)

# Train / test split (80 / 20)
split = int(0.8 * len(raw_data))
train_data = raw_data[:split]
test_data  = raw_data[split:]

print(f'Train samples after augmentation : {len(train_data)}')
print(f'Test  samples after augmentation : {len(test_data)}')
print(f'Total samples after augmentation : {len(raw_data)}')

# ── Simple whitespace tokeniser ───────────────────────────────────────────────
def tokenize(text: str):
    return text.lower().split()

# ── Build vocabulary from training data only ──────────────────────────────────
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

counter = Counter()
for sentence, _ in train_data:
    counter.update(tokenize(sentence))

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word, _ in counter.most_common():
    vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)
PAD_IDX    = vocab[PAD_TOKEN]
UNK_IDX    = vocab[UNK_TOKEN]

print(f'\nVocabulary size : {VOCAB_SIZE}')
print(f'PAD index       : {PAD_IDX}')
print(f'UNK index       : {UNK_IDX}')

# ── Encode sentence → list of integer IDs ────────────────────────────────────
def encode(sentence: str, vocab: dict) -> list:
    return [vocab.get(tok, UNK_IDX) for tok in tokenize(sentence)]

# ── Pad / truncate to fixed length ────────────────────────────────────────────
MAX_LEN = 20

def pad_sequence(ids: list, max_len: int, pad_idx: int) -> list:
    ids = ids[:max_len]                        # truncate
    ids += [pad_idx] * (max_len - len(ids))    # right-pad
    return ids

# ── PyTorch Dataset & DataLoader ──────────────────────────────────────────────
class SentimentDataset(Dataset):
    """Wraps raw (sentence, label) pairs into a torch Dataset."""

    def __init__(self, data, vocab, max_len):
        self.samples = []
        for sentence, label in data:
            ids    = encode(sentence, vocab)
            padded = pad_sequence(ids, max_len, PAD_IDX)
            self.samples.append((
                torch.tensor(padded, dtype=torch.long),
                torch.tensor(label,  dtype=torch.long)
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


BATCH_SIZE = 8

train_dataset = SentimentDataset(train_data, vocab, MAX_LEN)
test_dataset  = SentimentDataset(test_data,  vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# Inspect one batch
x_batch, y_batch = next(iter(train_loader))
print(f'\nInput batch shape  : {x_batch.shape}   (batch, seq_len)')
print(f'Label batch shape  : {y_batch.shape}')
print(f'Labels             : {y_batch.tolist()}')

Train samples after augmentation : 200
Test  samples after augmentation : 50
Total samples after augmentation : 250

Vocabulary size : 315
PAD index       : 0
UNK index       : 1

Input batch shape  : torch.Size([8, 20])   (batch, seq_len)
Label batch shape  : torch.Size([8])
Labels             : [1, 1, 1, 0, 0, 0, 1, 0]


In [3]:
# Cell 3: Scaled Dot-Product Attention
"""$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q** (Query), **K** (Key), **V** (Value) are linear projections of the input.
- Dividing by $\sqrt{d_k}$ prevents the dot products from growing too large, which would push softmax into regions with tiny gradients.
- The softmax output is a distribution over positions — it tells the model *how much attention* to pay to each token.
"""
def scaled_dot_product_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    mask: torch.Tensor = None
):
    """
    Compute scaled dot-product attention.

    Args:
        Q   : (..., seq_len_q, d_k)
        K   : (..., seq_len_k, d_k)
        V   : (..., seq_len_k, d_v)
        mask: optional boolean mask; True positions are IGNORED (set to -inf)

    Returns:
        output  : (..., seq_len_q, d_v)  — weighted sum of values
        weights : (..., seq_len_q, seq_len_k)  — attention weights
    """
    d_k = Q.size(-1)

    # Step 1: dot product of Q and K^T, then scale
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    # scores shape: (..., seq_len_q, seq_len_k)

    # Step 2: apply optional mask (e.g. padding mask)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # Step 3: softmax along the key dimension
    weights = F.softmax(scores, dim=-1)

    # Step 4: weighted sum of values
    output = torch.matmul(weights, V)

    return output, weights


# ── Sanity-check with random tensors ─────────────────────────────────────────
batch, seq, d_k, d_v = 2, 5, 16, 16
Q_test = torch.randn(batch, seq, d_k)
K_test = torch.randn(batch, seq, d_k)
V_test = torch.randn(batch, seq, d_v)

out, w = scaled_dot_product_attention(Q_test, K_test, V_test)
print(f'\nOutput shape  : {out.shape}   expected ({batch}, {seq}, {d_v})')
print(f'Weights shape : {w.shape}   expected ({batch}, {seq}, {seq})')
print(f'Weights sum   : {w[0].sum(dim=-1)}  (each row should sum to 1)')


Output shape  : torch.Size([2, 5, 16])   expected (2, 5, 16)
Weights shape : torch.Size([2, 5, 5])   expected (2, 5, 5)
Weights sum   : tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])  (each row should sum to 1)


In [4]:
# Cell 4: Multi-Head Self-Attention
"""Instead of a single attention, we project into `num_heads` subspaces, run attention in parallel, then concatenate and project back.

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1,\ldots,\text{head}_h)\,W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q,\, KW_i^K,\, VW_i^V)$$

This lets the model attend to information from different representation subspaces simultaneously.
"""
class MultiHeadSelfAttention(nn.Module):
    """
    Multi-Head Self-Attention module.

    Args:
        embed_dim  : total embedding dimension (must be divisible by num_heads)
        num_heads  : number of parallel attention heads
    """

    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        assert embed_dim % num_heads == 0, \
            f'embed_dim ({embed_dim}) must be divisible by num_heads ({num_heads})'

        self.embed_dim  = embed_dim
        self.num_heads  = num_heads
        self.head_dim   = embed_dim // num_heads   # d_k per head

        # Linear projections for Q, K, V and output
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=False)

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(B, S, E) → (B, H, S, head_dim)"""
        B, S, E = x.shape
        x = x.view(B, S, self.num_heads, self.head_dim)
        return x.permute(0, 2, 1, 3)           # (B, H, S, head_dim)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x    : (B, S, E)  — input embeddings
            mask : (B, 1, 1, S)  — optional padding mask
        Returns:
            out  : (B, S, E)
        """
        # Project inputs to Q, K, V
        Q = self.split_heads(self.W_q(x))      # (B, H, S, head_dim)
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        # Scaled dot-product attention for all heads in parallel
        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask)
        # attn_out: (B, H, S, head_dim)

        # Concatenate heads: (B, H, S, head_dim) → (B, S, E)
        B, H, S, _ = attn_out.shape
        attn_out = attn_out.permute(0, 2, 1, 3).contiguous().view(B, S, self.embed_dim)

        # Final linear projection
        out = self.W_o(attn_out)
        return out


# ── Quick test ────────────────────────────────────────────────────────────────
mha = MultiHeadSelfAttention(embed_dim=32, num_heads=4)
x_test = torch.randn(2, 10, 32)   # (batch=2, seq=10, embed=32)
out_test = mha(x_test)
print(f'\nMHA output shape : {out_test.shape}   expected (2, 10, 32)')


MHA output shape : torch.Size([2, 10, 32])   expected (2, 10, 32)


In [5]:
# Cell 5: Positional Encoding
"""Transformers are permutation-invariant — they have no notion of order. We inject position information using sinusoidal functions:

$$PE_{(pos, 2i)}   = \sin\!\left(pos / 10000^{2i/d_{\text{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\!\left(pos / 10000^{2i/d_{\text{model}}}\right)$$

Different frequencies allow the model to learn relative positions across any sequence length.
"""
class PositionalEncoding(nn.Module):
    """
    Fixed (non-learned) sinusoidal positional encoding.

    Args:
        embed_dim : embedding dimension
        max_len   : maximum sequence length supported
        dropout   : dropout probability
    """

    def __init__(self, embed_dim: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Build the positional encoding table once and register as buffer
        pe = torch.zeros(max_len, embed_dim)          # (max_len, embed_dim)

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2, dtype=torch.float) *
            (-math.log(10000.0) / embed_dim)
        )

        pe[:, 0::2] = torch.sin(position * div_term)  # even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)  # odd  dimensions

        pe = pe.unsqueeze(0)                           # (1, max_len, embed_dim)
        self.register_buffer('pe', pe)                 # not a learnable parameter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (B, S, embed_dim)
        Returns x + positional encoding for the first S positions.
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ── Visualise the positional encoding matrix ─────────────────────────────────
# This visualization will be performed in the final evaluation cell to reduce clutter.
# pe_vis = PositionalEncoding(embed_dim=64, max_len=50)
# pe_matrix = pe_vis.pe.squeeze(0).detach().numpy()   # (50, 64)
# print('Positional encoding shape:', pe_matrix.shape)


In [6]:
# Cell 6: Feed-Forward Network (FFN)
"""Each encoder block contains a position-wise FFN applied identically to each token:

$$\text{FFN}(x) = \max(0,\, xW_1 + b_1)\,W_2 + b_2$$

The inner dimension `ff_dim` is typically 4× the embedding dimension, giving the network extra capacity to transform representations.
"""
class FeedForwardNetwork(nn.Module):
    """
    Position-wise Feed-Forward Network.

    Args:
        embed_dim : input/output dimension
        ff_dim    : inner (hidden) dimension, typically 4 × embed_dim
        dropout   : dropout probability
    """

    def __init__(self, embed_dim: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, S, embed_dim) → (B, S, embed_dim)"""
        return self.net(x)


# ── Quick test ────────────────────────────────────────────────────────────────
ffn = FeedForwardNetwork(embed_dim=32, ff_dim=128)
x_test = torch.randn(2, 10, 32)
print(f'\nFFN output shape : {ffn(x_test).shape}   expected (2, 10, 32)')


FFN output shape : torch.Size([2, 10, 32])   expected (2, 10, 32)


In [7]:
# Cell 7: Transformer Encoder Block
"""A single encoder block combines:
1. Multi-Head Self-Attention
2. Add & Norm (residual connection + LayerNorm)
3. Feed-Forward Network
4. Add & Norm again

Residual connections help gradients flow through deep networks. LayerNorm stabilises training by normalising activations.
"""
class TransformerEncoderBlock(nn.Module):
    """
    Single Transformer Encoder block.

    Sub-layers:
        1.  Multi-Head Self-Attention + residual + LayerNorm
        2.  Feed-Forward Network      + residual + LayerNorm

    Args:
        embed_dim : embedding dimension
        num_heads : number of attention heads
        ff_dim    : FFN inner dimension
        dropout   : dropout probability
    """

    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()

        self.attention = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn       = FeedForwardNetwork(embed_dim, ff_dim, dropout)

        self.norm1   = nn.LayerNorm(embed_dim)
        self.norm2   = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x    : (B, S, embed_dim)
            mask : optional padding mask
        Returns:
            (B, S, embed_dim)
        """
        # Sub-layer 1: Multi-Head Attention + residual + LayerNorm
        attn_out = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_out))    # Add & Norm

        # Sub-layer 2: FFN + residual + LayerNorm
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)                   # Add & Norm

        return x


# ── Quick test ────────────────────────────────────────────────────────────────
enc_block = TransformerEncoderBlock(embed_dim=32, num_heads=4, ff_dim=128)
x_test    = torch.randn(2, 10, 32)
print(f'\nEncoder block output shape : {enc_block(x_test).shape}   expected (2, 10, 32)')


Encoder block output shape : torch.Size([2, 10, 32])   expected (2, 10, 32)


In [8]:
# Cell 8: Transformer Encoder Classifier Definition and Instantiation
"""We stack `num_layers` encoder blocks, mean-pool over the sequence, then pass through a linear classifier.
"""
class TransformerClassifier(nn.Module):
    """
    Text classifier built on a stacked Transformer Encoder.

    Architecture:
        Embedding → Positional Encoding
            → N × TransformerEncoderBlock
            → Mean Pooling
            → Linear Classifier

    Args:
        vocab_size  : vocabulary size (for the embedding table)
        embed_dim   : embedding dimension
        num_heads   : number of attention heads per block
        ff_dim      : FFN inner dimension per block
        num_layers  : number of stacked encoder blocks
        num_classes : number of output classes
        max_len     : maximum sequence length
        dropout     : dropout probability
        pad_idx     : index of <PAD> token (masked out in attention)
    """

    def __init__(
        self,
        vocab_size:  int,
        embed_dim:   int,
        num_heads:   int,
        ff_dim:      int,
        num_layers:  int,
        num_classes: int,
        max_len:     int  = 512,
        dropout:     float = 0.1,
        pad_idx:     int  = 0,
    ):
        super().__init__()
        self.pad_idx   = pad_idx
        self.embed_dim = embed_dim

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_enc   = PositionalEncoding(embed_dim, max_len, dropout)

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes),
        )

    def make_pad_mask(self, x: torch.Tensor) -> torch.Tensor:
        """Build a boolean mask: 1 for real tokens, 0 for PAD."""
        # x : (B, S)
        mask = (x != self.pad_idx).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, S)
        return mask.to(x.device)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x    : (B, S)  — integer token IDs
        Returns logits of shape (B, num_classes)
        """
        mask = self.make_pad_mask(x)           # (B, 1, 1, S)

        # 1. Token embedding + scale + positional encoding
        out = self.embedding(x) * math.sqrt(self.embed_dim)   # (B, S, E)
        out = self.pos_enc(out)

        # 2. Pass through each encoder block
        for layer in self.encoder_layers:
            out = layer(out, mask)

        # 3. Mean pooling over non-PAD positions
        pad_expanded = (x != self.pad_idx).unsqueeze(-1).float()  # (B, S, 1)
        out = (out * pad_expanded).sum(dim=1) / pad_expanded.sum(dim=1).clamp(min=1)
        # out : (B, E)

        # 4. Classify
        return self.classifier(out)            # (B, num_classes)


# ── Hyper-parameters ─────────────────────────────────────────────────────────
EMBED_DIM   = 64
NUM_HEADS   = 4
FF_DIM      = 256
NUM_LAYERS  = 2
NUM_CLASSES = 2
DROPOUT     = 0.3

model = TransformerClassifier(
    vocab_size  = VOCAB_SIZE,
    embed_dim   = EMBED_DIM,
    num_heads   = NUM_HEADS,
    ff_dim      = FF_DIM,
    num_layers  = NUM_LAYERS,
    num_classes = NUM_CLASSES,
    max_len     = MAX_LEN,
    dropout     = DROPOUT,
    pad_idx     = PAD_IDX,
).to(device)

print('\n' + '=' * 55)
print('         TRANSFORMER CLASSIFIER — ARCHITECTURE')
print('=' * 55)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Vocabulary size      : {VOCAB_SIZE}')
print(f'  Embedding dimension  : {EMBED_DIM}')
print(f'  Attention heads      : {NUM_HEADS}')
print(f'  Head dimension       : {EMBED_DIM // NUM_HEADS}')
print(f'  FFN inner dimension  : {FF_DIM}')
print(f'  Encoder layers       : {NUM_LAYERS}')
print(f'  Max sequence length  : {MAX_LEN}')
print(f'  Dropout              : {DROPOUT}')
print(f'  Output classes       : {NUM_CLASSES}')
print('-' * 55)
print(f'  Trainable parameters : {total_params:,}')
print('=' * 55)

# Verify forward pass shape
dummy_input  = torch.zeros(4, MAX_LEN, dtype=torch.long).to(device)
dummy_output = model(dummy_input)
print(f'\nForward pass: input {list(dummy_input.shape)} → output {list(dummy_output.shape)}')



         TRANSFORMER CLASSIFIER — ARCHITECTURE
  Vocabulary size      : 315
  Embedding dimension  : 64
  Attention heads      : 4
  Head dimension       : 16
  FFN inner dimension  : 256
  Encoder layers       : 2
  Max sequence length  : 20
  Dropout              : 0.3
  Output classes       : 2
-------------------------------------------------------
  Trainable parameters : 121,762

Forward pass: input [4, 20] → output [4, 2]


In [ ]:
# Cell 9: Training Setup and Loop
LEARNING_RATE = 3e-4
EPOCHS        = 200

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

# Learning-rate scheduler: reduce LR by 0.5 if val loss plateaus for 5 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print(f'\nCriterion  : {criterion}')
print(f'Optimiser  : Adam  (lr={LEARNING_RATE}, weight_decay=1e-5)')
print(f'Scheduler  : ReduceLROnPlateau (factor=0.5, patience=5)')
print(f'Epochs     : {EPOCHS}')

def train_one_epoch(model, loader, criterion, optimizer, device):
    """Run one full pass over the training set. Returns (avg_loss, accuracy)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(x_batch)                # (B, num_classes)
        loss   = criterion(logits, y_batch)

        loss.backward()
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * x_batch.size(0)
        preds       = logits.argmax(dim=-1)
        correct    += (preds == y_batch).sum().item()
        total      += x_batch.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate on a dataloader. Returns (avg_loss, accuracy)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits      = model(x_batch)
        loss        = criterion(logits, y_batch)

        total_loss += loss.item() * x_batch.size(0)
        preds       = logits.argmax(dim=-1)
        correct    += (preds == y_batch).sum().item()
        total      += x_batch.size(0)

    return total_loss / total, correct / total


print('\nStarting training loop...')

train_losses, train_accs = [], []
test_losses,  test_accs  = [], []

print(f'\n{"Epoch":>6}  {"Train Loss":>11}  {"Train Acc":>10}  {"Test Loss":>10}  {"Test Acc":>9}')
print('-' * 58)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)

    train_losses.append(tr_loss);  train_accs.append(tr_acc)
    test_losses.append(te_loss);   test_accs.append(te_acc)

    scheduler.step(te_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:>6}  {tr_loss:>11.4f}  {tr_acc*100:>9.1f}%  {te_loss:>10.4f}  {te_acc*100:>8.1f}%')

print('-' * 58)
print(f'Final train accuracy : {train_accs[-1]*100:.1f}%')
print(f'Final test  accuracy : {test_accs[-1]*100:.1f}%')



Criterion  : CrossEntropyLoss()
Optimiser  : Adam  (lr=0.0003, weight_decay=1e-5)
Scheduler  : ReduceLROnPlateau (factor=0.5, patience=5)
Epochs     : 200

Starting training loop...

 Epoch   Train Loss   Train Acc   Test Loss   Test Acc
----------------------------------------------------------
     1       0.6929       53.5%      0.6792      58.0%
     5       0.5125       81.0%      0.4982      82.0%
    10       0.3254       86.0%      0.2960      88.0%
    15       0.2535       90.0%      0.4031      86.0%
    20       0.1857       93.5%      0.2240      94.0%
    25       0.2181       92.5%      0.4689      84.0%
    30       0.1289       95.5%      0.3083      92.0%
    35       0.0483       97.5%      0.3509      92.0%
    40       0.0800       98.0%      0.3441      92.0%
    45       0.1097       96.5%      0.3565      92.0%
    50       0.2325       94.5%      0.3457      92.0%
    55       0.0641       97.5%      0.3551      92.0%
    60       0.0477       98.0%      0.363

: 

In [ ]:
# Cell 10: Evaluation, Predictions, and Visualizations
# ── Plot Training Loss and Accuracy ──────────────────────────────────────────
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Loss ─────────────────────────────────────────────────────────────────────
axes[0].plot(epochs_range, train_losses, label='Train Loss', color='steelblue',  linewidth=2)
axes[0].plot(epochs_range, test_losses,  label='Test Loss',  color='coral',      linewidth=2, linestyle='--')
axes[0].set_title('Loss vs Epoch', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Accuracy ─────────────────────────────────────────────────────────────────
axes[1].plot(epochs_range, [a*100 for a in train_accs], label='Train Accuracy', color='steelblue', linewidth=2)
axes[1].plot(epochs_range, [a*100 for a in test_accs],  label='Test Accuracy',  color='coral',     linewidth=2, linestyle='--')
axes[1].set_title('Accuracy vs Epoch', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(0, 105)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Transformer Encoder — Training Curves', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Example Predictions ──────────────────────────────────────────────────────
@torch.no_grad()
def predict(sentence: str, model, vocab, max_len, device) -> dict:
    """Predict sentiment for a single sentence."""
    model.eval()
    ids    = encode(sentence, vocab)
    padded = pad_sequence(ids, max_len, PAD_IDX)
    x      = torch.tensor([padded], dtype=torch.long).to(device)

    logits = model(x)                         # (1, num_classes)
    probs  = F.softmax(logits, dim=-1)[0]
    pred   = logits.argmax(dim=-1).item()

    return {
        'sentence'   : sentence,
        'prediction' : 'Positive ✅' if pred == 1 else 'Negative ❌',
        'confidence' : f'{probs[pred].item()*100:.1f}%',
        'p_positive' : f'{probs[1].item()*100:.1f}%',
        'p_negative' : f'{probs[0].item()*100:.1f}%',
    }


test_sentences = [
    "This was an absolutely wonderful experience I loved it",
    "Terrible product broke immediately very disappointed",
    "The movie was fantastic and the acting was superb",
    "Worst service ever I will never come back",
    "Amazing quality highly recommend to everyone",
    "Horrible experience completely ruined my day",
]

print('\n' + '=' * 68)
print('  EXAMPLE PREDICTIONS')
print('=' * 68)
for sent in test_sentences:
    result = predict(sent, model, vocab, MAX_LEN, device)
    print(f"  Sentence   : {result['sentence']}")
    print(f"  Prediction : {result['prediction']}   Confidence: {result['confidence']}")
    print(f"  P(pos)={result['p_positive']}  P(neg)={result['p_negative']}")
    print('-' * 68)

# ── Visualize Attention Weights ──────────────────────────────────────────────
@torch.no_grad()
def get_attention_weights(sentence: str, model, vocab, max_len, device):
    """Return attention weights from the first encoder block, first head."""
    model.eval()
    tokens = tokenize(sentence)
    ids    = [vocab.get(t, UNK_IDX) for t in tokens]
    padded = pad_sequence(ids, max_len, PAD_IDX)
    x      = torch.tensor([padded], dtype=torch.long).to(device)

    mask   = model.make_pad_mask(x)
    emb    = model.embedding(x) * math.sqrt(model.embed_dim)
    emb    = model.pos_enc(emb)                       # (1, S, E)

    # Pull Q, K from first encoder block, split into heads
    block = model.encoder_layers[0]
    Q = block.attention.split_heads(block.attention.W_q(emb))
    K = block.attention.split_heads(block.attention.W_k(emb))

    d_k    = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (1, H, S, S)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)               # (1, H, S, S)

    return tokens, weights[0, 0, :len(tokens), :len(tokens)].cpu().numpy()


sample_sentence = "I absolutely loved this movie it was fantastic"
tokens, attn = get_attention_weights(sample_sentence, model, vocab, MAX_LEN, device)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(attn, cmap='Blues', vmin=0, vmax=attn.max())
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(tokens, fontsize=10)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Attention Weights — Layer 1, Head 1', fontsize=12, fontweight='bold')
ax.set_xlabel('Key (attended to)')
ax.set_ylabel('Query (attending from)')
plt.tight_layout()
plt.show()

print(f'\nSentence: "{sample_sentence}"')
print('Brighter cells = higher attention weight between the two tokens.')

# ── Final Test-Set Evaluation Report ─────────────────────────────────────────
@torch.no_grad()
def full_evaluation(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    for x_batch, y_batch in loader:
        logits = model(x_batch.to(device))
        preds  = logits.argmax(dim=-1).cpu().tolist()
        all_preds  += preds
        all_labels += y_batch.tolist()

    # Confusion matrix
    TP = sum(p == 1 and l == 1 for p, l in zip(all_preds, all_labels))
    TN = sum(p == 0 and l == 0 for p, l in zip(all_preds, all_labels))
    FP = sum(p == 1 and l == 0 for p, l in zip(all_preds, all_labels))
    FN = sum(p == 0 and l == 1 for p, l in zip(all_preds, all_labels))

    accuracy  = (TP + TN) / len(all_labels)
    precision = TP / (TP + FP + 1e-9)
    recall    = TP / (TP + FN + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)

    return dict(
        accuracy=accuracy, precision=precision, recall=recall, f1=f1,
        TP=TP, TN=TN, FP=FP, FN=FN,
        all_preds=all_preds, all_labels=all_labels
    )


metrics = full_evaluation(model, test_loader, device)

print('\n' + '=' * 42)
print('   FINAL TEST-SET EVALUATION')
print('=' * 42)
print(f"  Accuracy  : {metrics['accuracy']*100:.1f}%")
print(f"  Precision : {metrics['precision']*100:.1f}%")
print(f"  Recall    : {metrics['recall']*100:.1f}%")
print(f"  F1 Score  : {metrics['f1']:.4f}")
print('\n  Confusion Matrix:')
print(f"           Pred NEG   Pred POS")
print(f"  True NEG    {metrics['TN']:4d}       {metrics['FP']:4d}")
print(f"  True POS    {metrics['FN']:4d}       {metrics['TP']:4d}")
print('=' * 42)

# ── Visualise confusion matrix ────────────────────────────────────────────────
cm = [[metrics['TN'], metrics['FP']],
      [metrics['FN'], metrics['TP']]]

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]);  ax.set_xticklabels(['Pred NEG', 'Pred POS'])
ax.set_yticks([0, 1]);  ax.set_yticklabels(['True NEG', 'True POS'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i][j], ha='center', va='center', fontsize=14, fontweight='bold',
                color='white' if cm[i][j] > max(metrics['TP'], metrics['TN']) / 2 else 'black')
ax.set_title('Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

print('\n' + '=' * 42)
print('   HOW ATTENTION HELPS THE MODEL')
print('=' * 42)
print("### What Scaled Dot-Product Attention Does\nAttention computes a **weighted sum of value vectors**, where the weights reflect *how relevant each token is to every other token*. For a query token at position *i*, the model scores its compatibility with all key tokens and uses those scores (after softmax) to combine value representations.\n\n### Why It Helps for Text Classification\n| Problem without attention | How attention solves it |\n|---|---|\n| RNNs read left-to-right and can forget early tokens | Attention directly connects any two positions in O(1) |\n| Bag-of-words ignores word order & context | Each token's representation is shaped by its neighbours |\n| A single vector must encode the whole sequence | Multiple heads learn different types of dependencies |\n\n### Multi-Head Advantage\nDifferent heads can specialise:\n- Head 1 might link a negation word (\"not\") to the adjective it modifies\n- Head 2 might connect sentiment words (\"fantastic\", \"terrible\") to the subject\n- Head 3 might capture long-range dependencies across the sentence\n\n### Positional Encoding\nWithout positional encoding the model is bag-of-words. Sinusoidal PE injects *unique positional fingerprints* into every token embedding, letting the model learn that \"not bad\" ≠ \"bad not\".\n\n### Residual Connections & LayerNorm\nResidual connections (`x = x + sublayer(x)`) prevent vanishing gradients, allowing deeper stacks. LayerNorm normalises activations per sample per layer, stabilising training.\n\n### Summary\nBy learning to **attend to the most sentiment-relevant words regardless of their position**, the Transformer encoder creates contextualised representations that are far richer than simple word-count features — which is why even a 2-layer, 64-dim model can distinguish positive from negative sentences effectively.")

# ── Save the Trained Model ───────────────────────────────────────────────────
torch.save({
    'model_state_dict' : model.state_dict(),
    'vocab'            : vocab,
    'hyperparams': {
        'vocab_size'  : VOCAB_SIZE,
        'embed_dim'   : EMBED_DIM,
        'num_heads'   : NUM_HEADS,
        'ff_dim'      : FF_DIM,
        'num_layers'  : NUM_LAYERS,
        'num_classes' : NUM_CLASSES,
        'max_len'     : MAX_LEN,
        'dropout'     : DROPOUT,
        'pad_idx'     : PAD_IDX,
    },
}, 'transformer_classifier.pth')

print('\nModel saved to transformer_classifier.pth')
print('\n=== All cells completed successfully ===')

: 